## Terms

- Lot: spaces for a component's most atomic entity to be placed (synonymous with "Homes" in residential parlance)
- Day: 1 iteration
- Month: 28 iterations
- Year: 12 * 28 iterations

## Demand target to demand valve

- All three RCI components' demand moves, moderated by a valve. 
- The demand-target is set monthly (each 28 iterations), and then the demand moves towards it slowly each day.
- Each component has a discrete rule for how it moves towards the target

## Lot increase/decrease

- New lots are introduced when the existing lots are 95% filled
- Existing lots are removed when they're 50% filled
- Number of lots added or removed is calculated weekly (7th iteration)
- 

## Residential pop increase

- Residential demand-target goes up when there's spare jobs available (commercial pop + industrial pop is greater than residential pop). Target is updated once per month.
- There's a "valve" effect, so the actual residential demand trails the demand-target. Trailing happens daily.
- Migration goes up when there's spare residential demand & down on the inverse.
- Residential demand is filled by migration (rate) + births (rate) - deaths (rate). This is applied daily. Capped by the number of available homes.
- A count of new homes between `0 to min(required, cap)` are added in each weekly iteration
- Residents are paid $1000 per month (if they're employed?)
- Residents consume daily

### Residential utility
- Residents have a marginal utility calculation, and consume as utility-maximising actors
- Residents consume from commercial output
- If the residents utility is unmet, they will migrate out

## Commercial pop increase
- Commercial demand-target goes up when residents marginal utility is under-maximised..? Calcualted monthly (28d)
- New commercial outlets open at a migration-rate
- Residents work at commercial outlets, and are paid by it.
- Commercial outlets make profit and loss, and if they make a loss, they close.
- A count of new lots between `0 to min(required, cap)` are added in each weekly iteration

### Commercial profit maximisation

- Commercial outlets are profit maximising entities
- If they run out of money, they go out of business, and their employees do not get paid

## Industrial pop increase
- All commercials are consumers of industry
- Industrial demand-target goes up when commercial 

### Industrial profit maximising
- Industrial outlets are profit maximising entities
- If thety run out of money, they go out of business, and their employees do not get paid

In [ ]:
from typing import Tuple
import math
import enum
from IPython.display import display
from utils.plotly_graphs.demand_graphs import plot_demand_bar, plot_demand_history

from utils.plotly_graphs.demand_graphs import plot_demand_detail

def check_period_end(i) -> Tuple[bool, bool, bool]:
    is_week_end = i % 7 == 0
    is_month_end = i % 28 == 0
    is_year_end = i % 336 == 0
    return is_week_end, is_month_end, is_year_end

def day_from_int(i: int) -> str:
    days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    return f'{i} ({days[(i - 1) % 7]})'

def month_from_int(i: int) -> str:
    months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    return f'{i} ({months[(i - 1) % 12]})'

history = {
    'residential_population': [],
    'commercial_population': [],
    'industrial_population': [],
    'residential_demand': [],
    'commercial_demand': [],
    'industrial_demand': [],
    'residential_demand_target': [],
    'commercial_demand_target': [],
    'industrial_demand_target': [],
    'residential_lots': [],
    'commercial_lots': [],
    'industrial_lots': [],
    'jobs_per_resident': [],
    'migration': [],
    'deaths': [],
    'births': [],
    'unemployment_rate': [],
}

class RciType(enum.Enum):
    RESIDENTIAL = 0
    COMMERCIAL = 1
    INDUSTRIAL = 2

class LotEntity:
    def __init__(self, id: int, lot_type: RciType):
        self.id = id
        self.type: RciType = lot_type
        self.occupied = False
        self.occupied_by: int | None = None

class ResidentEntity:
    def __init__(self, id: int, lot_address: int):
        self.id = id
        self.rci_type: RciType = RciType.RESIDENTIAL
        
        self.employed = False
        self.employed_at: int | None = None
        self.employed_type: RciType | None = None

        self.lot_address: int = lot_address

class CommercialEntity:
    def __init__(self, id: int, lot_address: int):
        self.id = id
        self.type: RciType = RciType.COMMERCIAL
        self.jobs = 1

        self.lot_address: int = lot_address


class IndustrialEntity:
    def __init__(self, id: int, lot_address: int):
        self.id = id
        self.type: RciType = RciType.INDUSTRIAL
        self.jobs = 1
        self.lot_address: int = lot_address

def main(iteration_count = 100):
    city_coffers = 0

    tax_rate_residential = 0.1
    tax_rate_commercial = 0.15
    tax_rate_industrial = 0.2

    residential_lots_cap = 10_000
    commercial_lots_cap = 5_000
    industrial_lots_cap = 5_000


    residential_demand_target = 0
    residential_demand = 0
    residential_population = 123
    resident_entities: list[ResidentEntity] = []

    commercial_demand_target = 0
    commercial_demand = 0
    commercial_population = 180
    commercial_entities: list[CommercialEntity] = []
    jobs_per_commercial_pop = 1

    industrial_demand_target = 0
    industrial_demand = 0
    industrial_population = 180
    industrial_entities: list[IndustrialEntity] = []
    jobs_per_industrial_pop = 1

    migration = 0

    lots: list[LotEntity] = []
    residential_lots_arr: list[LotEntity] = []
    commercial_lots_arr: list[LotEntity] = []
    industrial_lots_arr: list[LotEntity] = []

    birth_rate = 0.001
    death_rate = 0.0005

    day = 1
    week = 1
    month = 1
    year = 1950
    total_jobs = 0
    unfilled_jobs = 0
    unemployment_rate = 0

    def add_entity_to_lot(type: RciType, entity_id) -> bool:
        """
        Add an entity to the city if there's a vacant lot available

        Returns:
            - bool: True if an entity was added

        Raises:
            - ValueError: If there are no vacant lots available for the specified type
        """
        for lot in lots:
            if not lot.occupied and lot.type == type:
                lot.occupied = True
                lot.occupied_by = entity_id
                return True
        return False

    def add_lot(type: RciType) -> bool:
        """
        Add a lot to the city.
        
        Args:
            - type (RciType): The type of lot to add
        Returns:
            - bool: True if a lot was added
        """
        lot = LotEntity(len(lots), type)
        lots.append(lot)
        {
            RciType.RESIDENTIAL: residential_lots_arr,
            RciType.COMMERCIAL: commercial_lots_arr,
            RciType.INDUSTRIAL: industrial_lots_arr
        }[type].append(lot)


    def lots_of_type(type: RciType) -> list[LotEntity]:
            return [lot for lot in lots if lot.type == type]
    
    def add_entity_to_city(type: RciType) -> bool:
        """
        Add a new entity to the city.
        
        Create the new entity in the appropriate array, and add it to a lot if one exists.
        If a lot doesn't exist, raises an error.

        Args:
            - type (RciType): The type of entity to add (residential, commercial, or industrial)        
        Returns:
            - bool: True if an entity was added
        """
        next_id = 0
        if type == RciType.RESIDENTIAL:
            next_id = len(resident_entities) # todo: these id generations will overlap each other
            resident_entities.append(ResidentEntity(next_id, -1))
        if type == RciType.COMMERCIAL:
            next_id = len(commercial_entities) # todo: these id generations will overlap each other
            commercial_entities.append(CommercialEntity(next_id, -1))
        if type == RciType.INDUSTRIAL:
            next_id = len(industrial_entities) # todo: these id generations will overlap each other
            industrial_entities.append(IndustrialEntity(next_id, -1))

        added_to_lot = add_entity_to_lot(type, next_id)
        if not added_to_lot:
            raise ValueError(f"Failed to add {type.name.lower()} entity (id: {next_id}) to lot. len(lots[type]) is: {len(lots_of_type(type))}")
        return True


    def initialize_lots_and_pop_arrays():
        """
        Initialise lots and assign populations to each lot.

        In this model, a resident, commercial, or industrial occupies exactly one lot.
        """
        initial_residential_lots = int(residential_population * 1.35)
        initial_commercial_lots  = int(commercial_population * 1.35)
        initial_industrial_lots  = int(industrial_population * 1.35)

        for _ in range(initial_residential_lots):
            add_lot(RciType.RESIDENTIAL)
        for _ in range(initial_commercial_lots):
            add_lot(RciType.COMMERCIAL)
        for _ in range(initial_industrial_lots):
            add_lot(RciType.INDUSTRIAL)

        for _ in range(residential_population):
            add_entity_to_city(RciType.RESIDENTIAL)
        for _ in range(commercial_population):
            add_entity_to_city(RciType.COMMERCIAL)
        for _ in range(industrial_population):
            add_entity_to_city(RciType.INDUSTRIAL)

    initialize_lots_and_pop_arrays()
    

    for i in range(1, iteration_count + 1):
        is_week_end, is_month_end, is_year_end = check_period_end(i)
        print(f'--- ({i}) Day {day_from_int(day)}, Week {week}, Month {month_from_int(month)}, Year {year} ---')

        if is_week_end:
            print(f'It is the end of week {week}\n')
            week += 1
            day = 1
        else:
            day += 1
        if is_month_end:
            print(f'It is the end of month {month_from_int(month)} ({year}).\n')
            month += 1
            week = 1
        if is_year_end:
            print(f'It is the end of year {year}.\n')
            year += 1
            month = 1
        
        total_jobs = (commercial_population * jobs_per_commercial_pop) + (industrial_population * jobs_per_industrial_pop)
        unfilled_jobs = total_jobs - residential_population
        unemployment_rate = max(0, min(1, unfilled_jobs / total_jobs)) if total_jobs > 0 else 0
        print(f'Total Jobs: {total_jobs} for {residential_population} residents, Total Available Jobs: {unfilled_jobs}, Unemployment Rate: {unemployment_rate:.2%}')

        if is_month_end:
            residential_demand_target = max(-100, min(100, unfilled_jobs / 4))  # tune divisor
            if residential_demand_target < -100 or residential_demand_target > 100:
                raise ValueError(f"Residential demand target out of bounds: {residential_demand_target}")
            print(f'Total Jobs: {total_jobs} for {residential_population} residents, Total Available Jobs: {unfilled_jobs}')
            print(f'There are {unfilled_jobs} available jobs. Residential demand target will {"increase" if unfilled_jobs > 0 else "decrease"} to {residential_demand_target:.2f}')
        
        residential_demand_step = residential_demand_target - residential_demand
        residential_demand_rise_rate = 1
        residential_demand_fall_rate = 1
        residential_demand += max(-residential_demand_fall_rate, min(residential_demand_rise_rate, residential_demand_step))
        if residential_demand < -100 or residential_demand > 100:
            raise ValueError(f"Residential demand out of bounds: {residential_demand}")

        vacant_residential_lots = len(residential_lots_arr) - residential_population

        def calculate_births(residential_population: int, birth_rate: float) -> int:
            """
            Calculate the number of births in the city based on the residential population.

            Args:
                - residential_population (int): The current residential population of the city.
                - birth_rate (float): The birth rate to apply to the population.
            Returns:
                - int: The number of births to add to the city.
            """
            births = max(math.floor(residential_population * birth_rate), 0)
            return births
        
        def calculate_deaths(residential_population: int, death_rate: float) -> int:
            """
            Calculate the number of deaths in the city based on the residential population.

            Args:
                - residential_population (int): The current residential population of the city.
                - death_rate (float): The death rate to apply to the population.
            Returns:
                - int: The number of deaths to remove from the city.
            """
            deaths = max(math.floor(residential_population * death_rate), 0)
            return deaths


        def calculate_migration(residential_demand: float, residential_population: int) -> int:
            """
            Calculate the number of migrants to add or remove from the city based on residential demand.

            Args:
                - residential_demand (float): The current residential demand, ranging from -100 to 100.
                - residential_population (int): The current residential population of the city.
            Returns:
                - int: The number of migrants to add (positive) or remove (negative) from the city.
            """
            if residential_demand < 0:
                migration = max(math.ceil(residential_population * 0.005), 0) * -1
            elif residential_demand > 0:
                # we need at least one migrant when demand is positive, otherwise
                # the city gets stuck at 0 population and can't recover.
                migration = max(math.ceil(residential_population * 0.005), 1)
            else:
                migration = 0
            return migration
        
        births = calculate_births(residential_population, birth_rate)
        deaths = calculate_deaths(residential_population, death_rate)
        migration = calculate_migration(residential_demand, residential_population)

        projected_population_change = births - deaths + migration


        def adjust_vacant_lots(vacant_residential_lots: int, residential_lots_arr: list[LotEntity], residential_lots_cap: int):
            """
            Adjust the number of vacant residential lots based on the current population and lot capacity.

            Args:
                - vacant_residential_lots (int): The current number of vacant residential lots.
                - residential_lots_arr (list[LotEntity]): The list of current residential lots.
                - residential_lots_cap (int): The maximum number of residential lots allowed.
            """
            if vacant_residential_lots < len(residential_lots_arr) * 0.05:
                # if vacant_residential_lots is less than 5% of lots, add some lots
                for _ in range(10):
                    if len(residential_lots_arr) >= residential_lots_cap:
                        break
                    add_lot(RciType.RESIDENTIAL)
            elif vacant_residential_lots > len(residential_lots_arr) * 0.5:
                # else if vacant_residential_lots is more than 50% of lots, remove some lots
                removed = 0
                for lot in [l for l in residential_lots_arr if not l.occupied]:
                    if removed >=10 or len(residential_lots_arr) <= 100:
                        break
                    residential_lots_arr.remove(lot)
                    lots.remove(lot)
                    removed += 1

        # TODO, this `vacant_residential_lots` check stuff can be moved off somewhere else in the routine
        if is_week_end:
            adjust_vacant_lots(vacant_residential_lots, residential_lots_arr, residential_lots_cap)


        if (residential_population + projected_population_change) > len(residential_lots_arr):
            print(f"WARN: Cannot move residential population from {residential_population} to {residential_population + projected_population_change} with only {len(residential_lots_arr)} lots")

        if projected_population_change > 0:
            for _ in range(projected_population_change):
                # add_lot
                if not any (not l.occupied and l.type == RciType.RESIDENTIAL for l in lots):
                    pass
                    #add_lot(RciType.RESIDENTIAL)
                # if there are any vacant residential lots, add a resident to the city
                if any (not l.occupied and l.type == RciType.RESIDENTIAL for l in lots):
                    if add_entity_to_city(RciType.RESIDENTIAL):
                        residential_population += 1


        history['residential_population'].append(residential_population)
        history['commercial_population'].append(commercial_population)
        history['industrial_population'].append(industrial_population)
        history['residential_demand'].append(residential_demand)
        history['commercial_demand'].append(commercial_demand)
        history['industrial_demand'].append(industrial_demand)
        history['residential_demand_target'].append(residential_demand_target)
        history['commercial_demand_target'].append(commercial_demand_target)
        history['industrial_demand_target'].append(industrial_demand_target)
        history['residential_lots'].append(len(residential_lots_arr))
        history['commercial_lots'].append(len(commercial_lots_arr))
        history['industrial_lots'].append(len(industrial_lots_arr))
        history['jobs_per_resident'].append(total_jobs / residential_population if residential_population > 0 else 0)
        history['migration'].append(migration)
        history['deaths'].append(deaths)
        history['births'].append(births)
        history['unemployment_rate'].append(unemployment_rate)

    def plot_history():
        # fig_combined = plot_demand_history(
        #     residential_demand_history=history['residential_demand'],
        #     commercial_demand_history=history['commercial_demand'],
        #     industrial_demand_history=history['industrial_demand'],
        #     title="RCI Demand Over Time",
        # )
        # display(fig_combined)

        fig = plot_demand_detail(
            demand_history=history['residential_demand'],
            demand_target_history=history['residential_demand_target'],
            population_history=history['residential_population'],
            migration_history=history['migration'],
            births_history=history['births'],
            deaths_history=history['deaths'],
            lots_history=history['residential_lots'],
            title="Residential Detail",
            demand_color="rgb(76, 175, 80)",  # or use RESIDENTIAL_COLOR
        )
        display(fig)

        fig_res = plot_demand_history(
            residential_demand_history=history['residential_demand'],
            residential_target_history=history['residential_demand_target'],
            population_history=history['residential_population'],
            population_label="Residential Population",
            title="Residential Demand Over Time",
        )
        display(fig_res)

        fig_com = plot_demand_history(
            commercial_demand_history=history['commercial_demand'],
            commercial_target_history=history['commercial_demand_target'],
            population_history=history['commercial_population'],
            population_label="Commercial Population",
            title="Commercial Demand Over Time",
        )
        display(fig_com)

        fig_ind = plot_demand_history(
            industrial_demand_history=history['industrial_demand'],
            industrial_target_history=history['industrial_demand_target'],
            population_history=history['industrial_population'],
            population_label="Industrial Population",
            title="Industrial Demand Over Time",
        )
        display(fig_ind)
    plot_history()



main(1_000)